# Lesson 01 — PyTorch Deep Dive

Covers: custom `nn.Module`, `Dataset`/`DataLoader`, full training loop, torchscript basics.

**Interview relevance:** Expect live coding at this level — not just `model.fit()`.

## 1. Subclassing `nn.Module`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    """Fully-connected network with configurable depth."""
    def __init__(self, in_features, hidden_dims, out_features, dropout=0.3):
        super().__init__()
        dims = [in_features] + hidden_dims + [out_features]
        self.layers = nn.ModuleList(
            [nn.Linear(dims[i], dims[i+1]) for i in range(len(dims)-1)]
        )
        self.dropout = nn.Dropout(dropout)
        self.bn = nn.ModuleList(
            [nn.BatchNorm1d(d) for d in hidden_dims]
        )

    def forward(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.bn[i](x)
            x = F.relu(x)
            x = self.dropout(x)
        return self.layers[-1](x)   # logits — no activation on final layer

model = MLP(784, [512, 256], 10)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 2. Custom Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class TabularDataset(Dataset):
    """Generic dataset for numpy arrays."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Synthetic data
rng = np.random.default_rng(42)
X = rng.standard_normal((1000, 784))
y = rng.integers(0, 10, 1000)

dataset = TabularDataset(X, y)
loader  = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

xb, yb = next(iter(loader))
print(f"Batch shapes — X: {xb.shape}, y: {yb.shape}")


## 3. Full Training Loop (the right way)

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# --- setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP(784, [512, 256], 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = OneCycleLR(optimizer, max_lr=1e-2, steps_per_epoch=len(loader), epochs=5)

# split
n_val = 200
train_ds = TabularDataset(X[:-n_val], y[:-n_val])
val_ds   = TabularDataset(X[-n_val:], y[-n_val:])
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

for epoch in range(1, 6):
    tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_dl, criterion, device)
    scheduler.step()
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")


## 4. Saving & Loading Checkpoints

In [ ]:
import os

def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
    }, path)

def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt["epoch"]

save_checkpoint(model, optimizer, 5, "/tmp/model_ckpt.pt")
epoch_resumed = load_checkpoint(model, optimizer, "/tmp/model_ckpt.pt")
print(f"Resumed from epoch {epoch_resumed}")


## 5. TorchScript Export

In [ ]:
# TorchScript: compile model for production (no Python runtime needed)
scripted = torch.jit.script(model.cpu())
scripted.save("/tmp/model_scripted.pt")

loaded = torch.jit.load("/tmp/model_scripted.pt")
out = loaded(torch.randn(1, 784))
print("TorchScript output shape:", out.shape)


## Interview Q&A

**Q: Why `zero_grad()` before backward?**  
Gradients accumulate by default in PyTorch — calling `backward()` adds to `.grad`. Without zeroing, gradients from the previous batch corrupt the current update.

**Q: When would you use `no_grad()` vs `inference_mode()`?**  
`inference_mode()` is strictly stronger — it disables both gradient tracking AND version counters, giving ~10-15% speedup. Use it for inference. Use `no_grad()` when you still need tensor version info (e.g. some in-place ops during eval).

**Q: What does `clip_grad_norm_` do and why?**  
Rescales all gradients so their global L2 norm ≤ threshold. Prevents exploding gradients, especially in RNNs/transformers with long sequences.

**Q: `weight_decay` in AdamW vs Adam?**  
Adam applies weight decay as a gradient term (L2 reg), which interacts with adaptive scaling. AdamW decouples it — the decay is applied directly to weights, not through the gradient, which is the mathematically correct form and converges better.